# Scikit-learn for Machine Learning Students

A practical, student-friendly notebook covering the most useful **scikit-learn** tools and workflows for ML.

## Learning path
1. What scikit-learn is
2. Datasets and train/test split
3. Preprocessing and scaling
4. Encoding categorical data
5. Missing-value handling
6. Feature selection and dimensionality reduction
7. Regression
8. Classification
9. Clustering
10. Model evaluation
11. Cross-validation
12. Hyperparameter tuning
13. Pipelines
14. Feature engineering
15. Saving/loading models
16. Complete ML workflows

> **Core ML workflow:** Data → Split → Preprocess → Train → Predict → Evaluate → Tune → Save


In [3]:
# Install if needed:
# !pip install scikit-learn pandas numpy matplotlib seaborn joblib

import numpy as np
import pandas as pd

import sklearn
print("scikit-learn version:", sklearn.__version__)


scikit-learn version: 1.9.1


## 1. Built-in Datasets

Scikit-learn provides small datasets that are excellent for learning and experimentation.

In [4]:
from sklearn.datasets import (
    load_iris,
    load_diabetes,
    load_breast_cancer,
    load_digits,
    make_classification,
    make_regression,
    make_blobs
)

# Iris classification dataset
iris = load_iris()

print("Features shape:", iris.data.shape)
print("Target shape:", iris.target.shape)
print("Feature names:", iris.feature_names)
print("Target names:", iris.target_names)


Features shape: (150, 4)
Target shape: (150,)
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target names: ['setosa' 'versicolor' 'virginica']


In [5]:
# Convert a sklearn dataset into a DataFrame
iris_df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

iris_df["target"] = iris.target

display(iris_df.head())


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## 2. Train/Test Split

The most basic ML evaluation idea:

- Training data → model learns from it
- Test data → evaluate on unseen data

`random_state` makes the split reproducible.

In [6]:
from sklearn.model_selection import train_test_split

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


X_train: (120, 4)
X_test : (30, 4)
y_train: (120,)
y_test : (30,)


## 3. Feature Scaling

Many algorithms work better when numerical features are on comparable scales.

Common scalers:
- `StandardScaler` → mean 0, standard deviation 1
- `MinMaxScaler` → usually scales to 0–1
- `RobustScaler` → less sensitive to outliers

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training mean:", X_train_scaled.mean(axis=0))
print("Training std:", X_train_scaled.std(axis=0))


Training mean: [-1.20829273e-15 -2.03679666e-15  4.99600361e-16  1.67458640e-15]
Training std: [1. 1. 1. 1.]


### Important: `fit_transform()` vs `transform()`

Use:

```python
scaler.fit_transform(X_train)
```

for training data.

Use:

```python
scaler.transform(X_test)
```

for test data.

This prevents **data leakage** from the test set into preprocessing.

In [8]:
# Correct pattern
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Min-Max Scaling

In [9]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()

X_train_minmax = minmax.fit_transform(X_train)
X_test_minmax = minmax.transform(X_test)

print("Minimum:", X_train_minmax.min(axis=0))
print("Maximum:", X_train_minmax.max(axis=0))


Minimum: [0. 0. 0. 0.]
Maximum: [1. 1. 1. 1.]


## 5. Robust Scaling

In [10]:
from sklearn.preprocessing import RobustScaler

robust = RobustScaler()

X_train_robust = robust.fit_transform(X_train)
X_test_robust = robust.transform(X_test)


## 6. Missing Values

`SimpleImputer` can replace missing values using strategies such as:
- mean
- median
- most frequent
- constant

In [11]:
from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "Age": [20, 25, np.nan, 35, 40],
    "Salary": [40000, np.nan, 60000, 70000, 80000]
})

imputer = SimpleImputer(strategy="mean")

data_imputed = imputer.fit_transform(data)

print(data_imputed)


[[2.00e+01 4.00e+04]
 [2.50e+01 6.25e+04]
 [3.00e+01 6.00e+04]
 [3.50e+01 7.00e+04]
 [4.00e+01 8.00e+04]]


In [12]:
# Median is often useful when outliers exist
median_imputer = SimpleImputer(strategy="median")

data_median = median_imputer.fit_transform(data)

print(data_median)


[[2.0e+01 4.0e+04]
 [2.5e+01 6.5e+04]
 [3.0e+01 6.0e+04]
 [3.5e+01 7.0e+04]
 [4.0e+01 8.0e+04]]


## 7. Categorical Encoding

Machine-learning models usually require numerical input.

`OneHotEncoder` converts categories into binary columns.

In [13]:
from sklearn.preprocessing import OneHotEncoder

cat_data = pd.DataFrame({
    "Department": ["IT", "HR", "Finance", "IT", "HR"]
})

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded = encoder.fit_transform(cat_data)

print(encoded)
print("Categories:", encoder.categories_)


[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]
Categories: [array(['Finance', 'HR', 'IT'], dtype=object)]


## 8. Label Encoding

`LabelEncoder` is commonly used for a **target label**, not as a general replacement for one-hot encoding of input features.

In [14]:
from sklearn.preprocessing import LabelEncoder

labels = ["Spam", "Ham", "Spam", "Ham", "Spam"]

label_encoder = LabelEncoder()

encoded_labels = label_encoder.fit_transform(labels)

print("Encoded:", encoded_labels)
print("Classes:", label_encoder.classes_)

# Convert back
print("Decoded:", label_encoder.inverse_transform(encoded_labels))


Encoded: [1 0 1 0 1]
Classes: ['Ham' 'Spam']
Decoded: ['Spam' 'Ham' 'Spam' 'Ham' 'Spam']


## 9. ColumnTransformer

Different columns often need different preprocessing.

Example:
- numerical columns → StandardScaler
- categorical columns → OneHotEncoder

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.DataFrame({
    "Age": [20, 30, 40, 25],
    "Salary": [40000, 60000, 90000, 50000],
    "Department": ["IT", "HR", "IT", "Finance"]
})

X = df[["Age", "Salary", "Department"]]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Department"])
    ]
)

X_processed = preprocessor.fit_transform(X)

print(X_processed)


[[-1.18321596 -1.06904497  0.          0.          1.        ]
 [ 0.16903085  0.          0.          1.          0.        ]
 [ 1.52127766  1.60356745  0.          0.          1.        ]
 [-0.50709255 -0.53452248  1.          0.          0.        ]]


## 10. Regression: Linear Regression

Linear regression predicts a continuous numerical value.

Formula:

`ŷ = Xw + b`

In [16]:
from sklearn.linear_model import LinearRegression

X, y = make_regression(
    n_samples=200,
    n_features=3,
    noise=20,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("Predictions:", y_pred[:5])


Coefficients: [71.76924392 22.15973364 72.74381876]
Intercept: -1.2728990058172638
Predictions: [ 167.56002501   62.96832652  -52.49545916 -126.61833607   85.98654031]


## 11. Regression Metrics

Common regression metrics:
- MAE → average absolute error
- MSE → average squared error
- RMSE → square root of MSE
- R² → proportion of variance explained by the model

In [17]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)


MAE : 18.485676487850963
MSE : 530.4949095630434
RMSE: 23.032475107183842
R²  : 0.9420127985522943


## 12. Ridge Regression

Ridge adds **L2 regularization** to reduce overly large coefficients and help control overfitting.

In [18]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)

print("R²:", r2_score(y_test, ridge_pred))


R²: 0.9421595034262513


## 13. Lasso Regression

Lasso uses **L1 regularization** and can shrink some coefficients to exactly zero, which can help with feature selection.

In [19]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)

lasso.fit(X_train, y_train)

lasso_pred = lasso.predict(X_test)

print("R²:", r2_score(y_test, lasso_pred))
print("Coefficients:", lasso.coef_)


R²: 0.942166269365806
Coefficients: [71.65833711 22.02117867 72.64512707]


## 14. Logistic Regression

Despite its name, logistic regression is commonly used for **classification**.

For binary classification it models a probability using the sigmoid function.

In [20]:
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=5000)

log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)
y_prob = log_model.predict_proba(X_test_scaled)

print("Predictions:", y_pred[:10])
print("Probabilities:", y_prob[:5])


Predictions: [0 1 0 1 0 1 1 0 0 0]
Probabilities: [[9.99999941e-01 5.88824186e-08]
 [1.13351672e-05 9.99988665e-01]
 [9.93589175e-01 6.41082462e-03]
 [4.66491463e-01 5.33508537e-01]
 [9.99999999e-01 6.52500097e-10]]


## 15. Classification Metrics

In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy : 0.9824561403508771
Precision: 0.9861111111111112
Recall   : 0.9861111111111112
F1       : 0.9861111111111112

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

Confusion Matrix:
[[41  1]
 [ 1 71]]


## 16. Confusion Matrix Explained

For binary classification:

- True Positive (TP)
- True Negative (TN)
- False Positive (FP)
- False Negative (FN)

From these we derive accuracy, precision, recall, and F1.

In [22]:
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


TN: 41
FP: 1
FN: 1
TP: 71


## 17. ROC-AUC

ROC-AUC evaluates ranking quality using predicted probabilities.

For binary classification, use the probability of the positive class.

In [23]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob[:, 1])

print("ROC-AUC:", auc)


ROC-AUC: 0.9953703703703703


## 18. Decision Tree Classifier

Decision trees split data using feature-based rules.

In [24]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree.fit(X_train, y_train)

tree_pred = tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, tree_pred))


Accuracy: 0.9385964912280702


## 19. Random Forest

Random Forest combines many decision trees to form an ensemble model.

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))


Accuracy: 0.956140350877193


## 20. Feature Importance

Tree-based models can estimate the importance of input features.

In [26]:
importance = pd.Series(
    rf.feature_importances_,
    index=load_breast_cancer().feature_names
).sort_values(ascending=False)

display(importance.head(10))


worst area              0.138294
worst concave points    0.132993
worst radius            0.100805
mean concave points     0.098489
worst perimeter         0.072224
mean perimeter          0.068612
mean radius             0.067465
mean concavity          0.057445
mean area               0.050615
worst concavity         0.031544
dtype: float64

## 21. K-Nearest Neighbors (KNN)

KNN predicts based on nearby observations.

Because distance matters, scaling is usually important.

In [27]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, knn_pred))


Accuracy: 0.956140350877193


## 22. Support Vector Machine (SVM)

In [28]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm.fit(X_train_scaled, y_train)

svm_pred = svm.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, svm_pred))


Accuracy: 0.9824561403508771


/home/bakar/Documents/NAVTTC/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## 23. Naive Bayes

In [29]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, nb_pred))


Accuracy: 0.9385964912280702


## 24. Gradient Boosting

In [30]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, gb_pred))


Accuracy: 0.9473684210526315


## 25. Clustering with K-Means

K-Means is an **unsupervised learning** algorithm.

It groups observations into `k` clusters.

In [31]:
from sklearn.cluster import KMeans

X_blobs, _ = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=1.0,
    random_state=42
)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_blobs)

print("First 20 cluster labels:", clusters[:20])
print("Cluster centers:\n", kmeans.cluster_centers_)


First 20 cluster labels: [1 1 0 2 1 2 0 2 0 0 0 2 0 0 1 0 1 2 0 0]
Cluster centers:
 [[-2.63323268  9.04356978]
 [-6.88387179 -6.98398415]
 [ 4.74710337  2.01059427]]


## 26. K-Means Inertia

Inertia is the sum of squared distances of samples to their closest cluster center.

It is often examined with the **elbow method**.

In [32]:
inertias = []

for k in range(1, 10):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_blobs)
    inertias.append(model.inertia_)

print("Inertias:", inertias)


Inertias: [20402.34444292629, 5763.464789461438, 566.8595511244131, 496.4279529785782, 427.12865354980676, 375.03315834979685, 308.19836610418247, 272.4057314987481, 234.28072349591133]


## 27. Hierarchical / Agglomerative Clustering

In [33]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(
    n_clusters=3
)

agg_labels = agg.fit_predict(X_blobs)

print(agg_labels[:20])


[0 0 2 1 0 1 2 1 2 2 2 1 2 2 0 2 0 1 2 2]


## 28. Dimensionality Reduction with PCA

PCA reduces the number of features while trying to retain important variance.

In [34]:
from sklearn.decomposition import PCA

X, y = load_iris(return_X_y=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("Original shape:", X.shape)
print("PCA shape:", X_pca.shape)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())


Original shape: (150, 4)
PCA shape: (150, 2)
Explained variance ratio: [0.72962445 0.22850762]
Total explained variance: 0.9581320720000164


## 29. Cross-Validation

Instead of relying on one train/test split, cross-validation evaluates a model across multiple folds.

In [35]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

model = LogisticRegression(max_iter=5000)

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Fold scores:", scores)
print("Mean CV accuracy:", scores.mean())


Fold scores: [0.93859649 0.94736842 0.98245614 0.92982456 0.95575221]
Mean CV accuracy: 0.9507995652848935


## 30. K-Fold Cross-Validation

In [36]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = LogisticRegression(max_iter=5000)

scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="accuracy"
)

print("Scores:", scores)
print("Mean:", scores.mean())


Scores: [0.95614035 0.96491228 0.93859649 0.97368421 0.92035398]
Mean: 0.9507374631268437


## 31. Stratified K-Fold

For classification, `StratifiedKFold` attempts to preserve class proportions in each fold.

In [37]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    LogisticRegression(max_iter=5000),
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print("Scores:", scores)
print("Mean:", scores.mean())


Scores: [0.96491228 0.92105263 0.96491228 0.94736842 0.97345133]
Mean: 0.9543393882937432


## 32. Hyperparameter Tuning with GridSearchCV

Grid search tests specified combinations of hyperparameters.

In [38]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)


Best parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Best CV score: 0.9631268436578171


## 33. RandomizedSearchCV

Instead of testing every combination, randomized search samples a specified number of combinations.

In [39]:
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(random_state=42)

param_distributions = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10]
}

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

print("Best parameters:", random_search.best_params_)
print("Best score:", random_search.best_score_)


Best parameters: {'n_estimators': 50, 'min_samples_split': 5, 'max_depth': None}
Best score: 0.9631268436578171


## 34. Pipelines

A pipeline combines preprocessing and modeling into one reproducible workflow.

This is especially useful for avoiding preprocessing mistakes and data leakage.

In [40]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))


Accuracy: 0.9824561403508771


## 35. Pipeline + GridSearchCV

In [41]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)


Best parameters: {'model__C': 0.1}
Best CV score: 0.9802197802197803


## 36. Feature Selection with SelectKBest

In [42]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(
    score_func=f_classif,
    k=5
)

X_selected = selector.fit_transform(X_train, y_train)

print("Original features:", X_train.shape[1])
print("Selected features:", X_selected.shape[1])


Original features: 30
Selected features: 5


## 37. Recursive Feature Elimination (RFE)

In [43]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

estimator = LogisticRegression(max_iter=5000)

rfe = RFE(
    estimator=estimator,
    n_features_to_select=5
)

rfe.fit(X_train_scaled, y_train)

print("Selected:", rfe.support_)
print("Ranking:", rfe.ranking_)


Selected: [False False False False False False False False False False  True False
 False False False False False False False False  True  True False  True
 False False False  True False False]
Ranking: [ 9 11 14  5 19 12 15  8 23 21  1 20 13  3 24  7 25 17 18 16  1  1  2  1
  4 26  6  1 10 22]


## 38. Polynomial Features

Polynomial features can allow a linear model to capture nonlinear relationships.

In [44]:
from sklearn.preprocessing import PolynomialFeatures

X_small = np.array([[1], [2], [3], [4]])

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_poly = poly.fit_transform(X_small)

print(X_poly)
print("Feature names:", poly.get_feature_names_out(["X"]))


[[ 1.  1.]
 [ 2.  4.]
 [ 3.  9.]
 [ 4. 16.]]
Feature names: ['X' 'X^2']


## 39. Learning Curves

Learning curves help study how model performance changes as the amount of training data increases.

In [45]:
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000)

train_sizes, train_scores, validation_scores = learning_curve(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 5)
)

print("Train sizes:", train_sizes)
print("Mean training scores:", train_scores.mean(axis=1))
print("Mean validation scores:", validation_scores.mean(axis=1))


Train sizes: [ 45 147 250 352 455]
Mean training scores: [1.         0.96870748 0.9672     0.96306818 0.96131868]
Mean validation scores: [0.84694923 0.93143922 0.95079957 0.94727527 0.95256948]


## 40. Prediction Probabilities

In [46]:
model = LogisticRegression(max_iter=5000)

model.fit(X_train_scaled, y_train)

probabilities = model.predict_proba(X_test_scaled)

print("First 5 probability rows:")
print(probabilities[:5])


First 5 probability rows:
[[9.99999941e-01 5.88824186e-08]
 [1.13351672e-05 9.99988665e-01]
 [9.93589175e-01 6.41082462e-03]
 [4.66491463e-01 5.33508537e-01]
 [9.99999999e-01 6.52500097e-10]]


## 41. Precision-Recall Curve

In [47]:
from sklearn.metrics import precision_recall_curve

y_scores = model.predict_proba(X_test_scaled)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_scores
)

print("Precision:", precision[:5])
print("Recall:", recall[:5])


Precision: [0.63157895 0.63716814 0.64285714 0.64864865 0.65454545]
Recall: [1. 1. 1. 1. 1.]


## 42. Regression Cross-Validation

In [48]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

X_reg, y_reg = make_regression(
    n_samples=300,
    n_features=5,
    noise=15,
    random_state=42
)

reg_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

scores = cross_val_score(
    reg_model,
    X_reg,
    y_reg,
    cv=5,
    scoring="r2"
)

print("R² scores:", scores)
print("Mean R²:", scores.mean())


R² scores: [0.93918078 0.91323881 0.92917923 0.91425525 0.87461167]
Mean R²: 0.914093147510936


## 43. Model Persistence

Save a trained model so it can be reused without retraining.

`joblib` is commonly used for scikit-learn objects.

In [49]:
import joblib

# Example:
model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)

joblib.dump(model, "logistic_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Models saved.")


Models saved.


In [50]:
# Load the saved model and scaler
loaded_model = joblib.load("logistic_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

new_data_scaled = loaded_scaler.transform(X_test[:3])
print("Loaded-model predictions:", loaded_model.predict(new_data_scaled))


ValueError: X has 30 features, but StandardScaler is expecting 4 features as input.

## 44. Complete ML Pipeline Example

This example demonstrates a professional-style workflow:

**Data → Split → Pipeline → Cross-validation → Tuning → Test evaluation**

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

search.fit(X_train, y_train)

final_pred = search.predict(X_test)

print("Best parameters:", search.best_params_)
print("Best CV score:", search.best_score_)
print("Test accuracy:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))


# 45. Algorithm Selection Cheat Sheet

| Problem | Algorithms to learn |
|---|---|
| Binary classification | Logistic Regression, KNN, SVM, Decision Tree, Random Forest |
| Multi-class classification | Logistic Regression, KNN, SVM, Trees, Random Forest, Gradient Boosting |
| Regression | Linear Regression, Ridge, Lasso, Decision Tree, Random Forest, Gradient Boosting |
| Clustering | K-Means, Agglomerative Clustering |
| Dimensionality reduction | PCA |
| Feature selection | SelectKBest, RFE |
| Nonlinear relationships | Trees, Random Forest, Gradient Boosting, SVM |
| High-dimensional data | Regularization, feature selection, PCA |


# 46. ML Workflow Cheat Sheet

```text
                 RAW DATA
                     ↓
             EDA / DATA CHECK
                     ↓
             Train / Test Split
                     ↓
          ┌──────────┴──────────┐
          ↓                     ↓
     Numerical             Categorical
     Imputation             Encoding
     Scaling
          └──────────┬──────────┘
                     ↓
                 Pipeline
                     ↓
              Model Training
                     ↓
              Cross-Validation
                     ↓
             Hyperparameter Tuning
                     ↓
                  Predict
                     ↓
                Evaluation
                     ↓
               Save Model
                     ↓
               Deployment
```

## Most important scikit-learn concepts for ML students

1. `train_test_split()`
2. `StandardScaler()`
3. `SimpleImputer()`
4. `OneHotEncoder()`
5. `ColumnTransformer()`
6. `Pipeline()`
7. `LinearRegression()`
8. `LogisticRegression()`
9. `DecisionTreeClassifier()`
10. `RandomForestClassifier()`
11. `KNeighborsClassifier()`
12. `SVC()`
13. `GradientBoostingClassifier()`
14. `KMeans()`
15. `PCA()`
16. `cross_val_score()`
17. `GridSearchCV()`
18. `RandomizedSearchCV()`
19. `classification_report()`
20. `confusion_matrix()`
21. `accuracy_score()`
22. `precision_score()`
23. `recall_score()`
24. `f1_score()`
25. `roc_auc_score()`

> **Key teaching point:** Students should understand the workflow and why each step is used, not just memorize algorithms.
